# ConFit Fine-Tuning — Notebook 2: Data Preparation

Builds the training-ready sentence-pair dataset in four steps:

| Step | What | Output |
|------|------|--------|
| 1 | Extract skill **evidence** sentences from resumes (Nvidia LLM) | `resume_evidence.jsonl` |
| 2 | Extract skill **requirement** sentences from JDs (Nvidia LLM) | `jd_requirements.jsonl` |
| 3 | Normalize all raw skill names → O*NET canonical IDs (Qdrant + Nvidia LLM, reuses `app/services/skill_normalizer`) | `skill_cache.json`, both normalized JSONLs |
| 4 | Build positive pairs + hard negatives | (NB3 input) |

**Hardware assumption:** local cluster with access to NVIDIA API + Qdrant Cloud + Neo4j Aura (same as production app).

**Datasets confirmed from NB1:**
- `Resume.csv` — 2,484 resumes, columns: `ID`, `Resume_str`, `Category` (24 UPPERCASE labels)
- `jobs.csv` — 2,277 JDs, columns: `Job Title`, `Job Description`

## Section 1 — Install Dependencies

In [3]:
%pip install -q kagglehub openai qdrant-client sentence-transformers neo4j nest_asyncio python-dotenv pandas tqdm loguru pip install ipywidgets redis

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install
You should consider upgrading via the 'c:\Users\USR005\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


## Section 2 — Imports, Paths & App Bootstrap

Adds the project root to `sys.path` and loads `.env` so the existing `app/` modules (skill_normalizer, embedding_client, vector_client, nvidia_llm_client) can be imported without modification.

In [1]:
import sys
import os
import json
import re
import time
import asyncio
import uuid
import warnings
from pathlib import Path

import nest_asyncio
import pandas as pd
import kagglehub
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── Project root resolution ──────────────────────────────────────────────────
# Notebook lives at  <root>/machine_learning/02_data_preparation.ipynb
# Project root is one level up.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change cwd to root so pydantic-settings can find .env
os.chdir(PROJECT_ROOT)

# ── Apply nest_asyncio so we can use asyncio.run() inside Jupyter ────────────
nest_asyncio.apply()

# ── Output directories ────────────────────────────────────────────────────────
OUT_DIR = NOTEBOOK_DIR / "data" / "nb2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_EVIDENCE_FILE  = OUT_DIR / "resume_evidence.jsonl"
JD_REQUIREMENTS_FILE  = OUT_DIR / "jd_requirements.jsonl"
SKILL_CACHE_FILE      = OUT_DIR / "skill_cache.json"
RESUME_NORM_FILE      = OUT_DIR / "resume_evidence_normalized.jsonl"
JD_NORM_FILE          = OUT_DIR / "jd_requirements_normalized.jsonl"

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs dir  : {OUT_DIR}")
print(f"sys.path[0]  : {sys.path[0]}")

Project root : c:\Users\USR005\OneDrive - Maersk Group\Documents\GitHub\IISC_emp_onboarding_proj
Outputs dir  : c:\Users\USR005\OneDrive - Maersk Group\Documents\GitHub\IISC_emp_onboarding_proj\machine_learning\data\nb2_outputs
sys.path[0]  : c:\Users\USR005\OneDrive - Maersk Group\Documents\GitHub\IISC_emp_onboarding_proj


In [22]:
# ── Import app modules (requires sys.path set above) ────────────────────────
# These reuse the exact same clients as the production app:
#   - nvidia_llm_client  → openai/gpt-oss-20b via NVIDIA API (async, streaming)
#   - embedding_client   → intfloat/multilingual-e5-small (local)
#   - vector_client      → Qdrant Cloud
#   - normalize_skill    → full 2-stage pipeline (Qdrant search + LLM judge + auto-upsert)

from app.clients.nvidia_llm_client import nvidia_llm_client
from app.clients.embedding_client  import embedding_client
from app.clients.vector_client     import vector_client
from app.services.skill_normalizer import normalize_skill
from app.config import settings

print(f"NVIDIA model        : {nvidia_llm_client.model}")
print(f"Embedding model     : {embedding_client.MODEL_NAME}")
print(f"Qdrant URL          : {settings.QDRANT_URL[:40]}...")
print(f"Qdrant connected    : {vector_client.test_connection()}")


NVIDIA model        : openai/gpt-oss-20b
Embedding model     : intfloat/multilingual-e5-small
Qdrant URL          : https://916e214e-e285-45c4-9296-b0f91c7c...
2026-03-21 11:43:08.306 | INFO     | app.clients.vector_client:test_connection:15 - Testing connection to Qdrant at https://916e214e-e285-45c4-9296-b0f91c7ca6c8.eu-west-2-0.aws.cloud.qdrant.io
2026-03-21 11:43:09.305 | INFO     | app.clients.vector_client:test_connection:18 - Qdrant connection successful. Found 1 collections.
Qdrant connected    : True


## Section 3 — Download & Load Datasets

Same kagglehub downloads as NB1 — cached locally after first run.

In [11]:
jobs_path   = kagglehub.dataset_download("kshitizregmi/jobs-and-job-description")
resume_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

jobs_df   = pd.read_csv(list(Path(jobs_path).rglob("*.csv"))[0])
resume_df = pd.read_csv(list(Path(resume_path).rglob("*.csv"))[0])

# ── Confirmed column names from NB1 exploration ──────────────────────────────
JOBS_TITLE_COL = "Job Title"
JOBS_TEXT_COL  = "Job Description"
RES_TEXT_COL   = "Resume_str"
RES_CAT_COL    = "Category"

# Drop nulls in critical columns
jobs_df   = jobs_df.dropna(subset=[JOBS_TITLE_COL, JOBS_TEXT_COL]).reset_index(drop=True)
resume_df = resume_df.dropna(subset=[RES_TEXT_COL, RES_CAT_COL]).reset_index(drop=True)

print(f"Jobs   : {len(jobs_df):,} rows")
print(f"Resumes: {len(resume_df):,} rows")
print(f"Resume categories ({resume_df[RES_CAT_COL].nunique()}): {sorted(resume_df[RES_CAT_COL].unique().tolist())}")

Jobs   : 2,277 rows
Resumes: 2,484 rows
Resume categories (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


## Section 4 — LLM Extraction Helpers

The existing `nvidia_llm_client` is async. We wrap it with `asyncio.run()` for use in notebook cells.
Prompts match `ml.md` specification exactly.

In [24]:

# ── Prompts ────────────────────────────────────────────────────────────────────
EVIDENCE_SYSTEM = (
    "You are a skill evidence extractor. "
    "Output ONLY a valid JSON array. No explanation, no preamble, no markdown. "
    "If the model provides reasoning, it is ignored; provide the final JSON in the 'content' field."
)

EVIDENCE_PROMPT = """\
Extract skill evidence from this resume.
For each skill, return the sentence that best shows HOW it was used.

Return ONLY this JSON array — nothing else:
[
  {{"skill": "<raw skill name>", "evidence": "<exact usage sentence>", "depth_signal": "surface|regular|deep|expert"}}
]

Rules:
- evidence must describe actual work, not just list the skill name
- if no context exists, omit the skill entirely
- maximum 10 skills

Resume:
{resume_text}
"""

REQUIREMENT_SYSTEM = (
    "You are a job requirement extractor. "
    "Output ONLY a valid JSON array. No explanation, no preamble, no markdown."
)

REQUIREMENT_PROMPT = """\
Extract skill requirements from this job description.
For each skill, return the sentence that best describes the expected proficiency.

Return ONLY this JSON array — nothing else:
[
  {{"skill": "<raw skill name>", "requirement": "<exact expectation sentence>", "level": "junior|mid|senior"}}
]

Rules:
- requirement must describe expected proficiency, not just name the skill
- if no context exists, omit the skill entirely
- maximum 10 skills

Job Description:
{jd_text}
"""

# ── Concurrency config ─────────────────────────────────────────────────────────
CONCURRENCY_LLM  = 5
CONCURRENCY_NORM = 5
NORM_CHUNK_SIZE  = 50

# ── Direct API call — bulletproof content extraction ──────────────────────────
async def _llm_content_only(system: str, user: str, max_tokens: int = 2048) -> str:
    """
    Calls NVIDIA API. For reasoning models, we explicitly take the 'content' field
    which usually duplicates the final result even if 'reasoning_content' is huge.
    """
    completion = await nvidia_llm_client.client.chat.completions.create(
        model=nvidia_llm_client.model, # Use .model instead of .MODEL
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0.0,
        stream=False, # Faster and easier to handle for this pipeline
    )
    
    # The 'content' field in these models usually contains the final answer 
    # stripped of the CoT/Reasoning.
    return (completion.choices[0].message.content or "").strip()

# ── JSON extraction: strip markdown fences and parse ──────────────────────────
_JSON_ARRAY_RE = re.compile(r"\[\s*\{.*\}\s*\]", re.DOTALL)

def extract_json_array(raw: str) -> list:
    """Pull the last JSON array out of an LLM reply."""
    if not raw:
        return []
        
    # 1. Strip markdown code blocks
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("`").strip()
    
    # 2. Try direct load (best case)
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list): return parsed
    except:
        pass
        
    # 3. Regex find (incase there's leftover text)
    matches = _JSON_ARRAY_RE.findall(raw)
    if matches:
        try:
            parsed = json.loads(matches[-1])
            if isinstance(parsed, list): return parsed
        except:
            pass
            
    return []

print(f"Prompts and helpers revised. Concurrency: LLM={CONCURRENCY_LLM}")
print("Using non-streaming calls for reliable content extraction.")


Prompts and helpers revised. Concurrency: LLM=5
Using non-streaming calls for reliable content extraction.


## Section 5 — Step 1: Extract Skill Evidence from Resumes

Processes each resume individually. Saves results incrementally to `resume_evidence.jsonl` — **resumable**: already-processed rows are skipped on re-run.# ── Filter for ENGINEERING and INFORMATION-TECHNOLOGY ────────────────────────TARGET_CATEGORIES = ["ENGINEERING", "INFORMATION-TECHNOLOGY"]# Filter resumesresume_df = resume_df[resume_df[RES_CAT_COL].isin(TARGET_CATEGORIES)].reset_index(drop=True)# Filter jobs (logic: match any target word in title or decription, or specific JD categories if they exist)
# For the jobs dataset, since categories aren't explicit, we'll filters for titles containing Eng, Tech, IT, or Soft.
jobs_df = jobs_df[
    jobs_df[JOBS_TITLE_COL].str.contains("Engineer|Technology|IT|Software|Developer|Data", case=False, na=False)
].reset_index(drop=True)

print(f"Filtered Jobs   : {len(jobs_df):,} rows")
print(f"Filtered Resumes: {len(resume_df):,} rows")
print(f"Resume categories: {resume_df[RES_CAT_COL].value_counts().to_dict()}")

# Drop nulls in critical columns
# (Already done but keeping for safety in filtered version)
jobs_df   = jobs_df.dropna(subset=[JOBS_TITLE_COL, JOBS_TEXT_COL]).reset_index(drop=True)
resume_df = resume_df.dropna(subset=[RES_TEXT_COL, RES_CAT_COL]).reset_index(drop=True)

print(f"Final Count - Jobs: {len(jobs_df):,}, Resumes: {len(resume_df):,}")

In [6]:
MAX_RESUME_WORDS = 1200

# ── Find already-processed resume IDs (resumability) ─────────────────────────
processed_resume_ids: set = set()
if RESUME_EVIDENCE_FILE.exists():
    with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                processed_resume_ids.add(obj["resume_id"])
            except Exception:
                pass
    print(f"Resuming: {len(processed_resume_ids)} resumes already processed.")
else:
    print("Starting fresh — no existing evidence file.")

Resuming: 2484 resumes already processed.


In [ ]:

# ── Async task: one resume → one JSONL record ─────────────────────────────────
async def _extract_one_resume(sem: asyncio.Semaphore, resume_id: str, category: str, resume_text: str) -> dict:
    words = resume_text.split()
    if len(words) > MAX_RESUME_WORDS:
        resume_text = " ".join(words[:MAX_RESUME_WORDS])
    async with sem:
        try:
            raw_reply = await _llm_content_only(
                system=EVIDENCE_SYSTEM,
                user=EVIDENCE_PROMPT.format(resume_text=resume_text),
            )
        except Exception as e:
            print(f"\n[!] Resume {resume_id} failed: {e}")
            raw_reply = "[]"
    skills = extract_json_array(raw_reply)
    valid_skills = [
        s for s in skills
        if isinstance(s, dict)
        and s.get("evidence")
        and s["evidence"].strip().lower() != s.get("skill", "").strip().lower()
    ]
    return {"resume_id": resume_id, "category": category, "skills": valid_skills}


# ── Async orchestrator — writes & flushes each record immediately ──────────────
async def run_resume_extraction(pending_df: pd.DataFrame, out_file) -> int:
    sem = asyncio.Semaphore(CONCURRENCY_LLM)
    tasks = [
        asyncio.ensure_future(
            _extract_one_resume(sem, str(r["ID"]), str(r[RES_CAT_COL]), str(r[RES_TEXT_COL]))
        )
        for _, r in pending_df.iterrows()
    ]
    count = 0
    pbar = tqdm(total=len(tasks), desc="Resumes (concurrent)")
    for coro in asyncio.as_completed(tasks):
        record = await coro
        out_file.write(json.dumps(record) + "\n")
        out_file.flush()   # ← persists to disk immediately; safe to interrupt
        count += 1
        pbar.update(1)
    pbar.close()
    return count

print("Functions defined: _extract_one_resume, run_resume_extraction (incremental write)")


Functions defined: _extract_one_resume, run_resume_extraction


In [31]:
# ── SINGLE-RESUME TEST — validates the exact pipeline before full run ──────────
test_row = pending_resume_df.iloc[0]

async def _test_one():
    sem = asyncio.Semaphore(1)
    return await _extract_one_resume(
        sem,
        str(test_row["ID"]),
        str(test_row[RES_CAT_COL]),
        str(test_row[RES_TEXT_COL]),
    )

result = asyncio.run(_test_one())

print(f"Resume ID  : {result['resume_id']}")
print(f"Category   : {result['category']}")
print(f"Skills found: {len(result['skills'])}\n")
for s in result["skills"]:
    print(f"  skill        : {s.get('skill')}")
    print(f"  evidence     : {str(s.get('evidence', ''))[:120]}")
    print(f"  depth_signal : {s.get('depth_signal')}")
    print()

if result["skills"]:
    print("✓ Looks good — safe to run full extraction in the next cell.")
else:
    print("✗ No skills extracted — check prompts or model response before full run.")

Resume ID  : 16852973
Category   : HR
Skills found: 10

  skill        : Human Resources
  evidence     : Helps to develop policies, directs and coordinates activities such as employment, compensation, labor relations, benefit
  depth_signal : regular

  skill        : Benefits
  evidence     : Administers benefits programs such as life, health, dental, insurance, pension plans, vacation, sick leave, leave of abs
  depth_signal : deep

  skill        : Marketing Collateral
  evidence     : Designed and created marketing collateral for sales meetings, trade shows and company executives.
  depth_signal : regular

  skill        : Advertising
  evidence     : Managed the in-house advertising program consisting of print and media collateral pieces.
  depth_signal : regular

  skill        : Website
  evidence     : Assisted in the complete design and launch of the company's website in 2 months.
  depth_signal : deep

  skill        : Medical Billing
  evidence     : Reviewed medical bills 

In [ ]:

# ── FULL RUN — only execute after test above passes ───────────────────────────
pending_resume_df = resume_df[~resume_df["ID"].astype(str).isin(processed_resume_ids)]
print(f"Pending: {len(pending_resume_df):,} resumes  |  Already done: {len(processed_resume_ids):,}")

if len(pending_resume_df) > 0:
    with open(RESUME_EVIDENCE_FILE, "a", encoding="utf-8") as out_f:
        count = asyncio.run(run_resume_extraction(pending_resume_df, out_f))
    print(f"\nDone. {count:,} resumes written.")
else:
    print("All resumes already processed — nothing to do.")

print(f"Output: {RESUME_EVIDENCE_FILE}")


Pending: 2,484 resumes  |  Already done: 0


Resumes (concurrent):   0%|          | 0/2484 [00:00<?, ?it/s]


Done. 2,484 resumes written.
Output: c:\Users\USR005\OneDrive - Maersk Group\Documents\GitHub\IISC_emp_onboarding_proj\machine_learning\data\nb2_outputs\resume_evidence.jsonl


In [27]:
# ── Inspect evidence output ───────────────────────────────────────────────────
evidence_flat = []  # one row per (resume, skill)
with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            evidence_flat.append({
                "resume_id":   rec["resume_id"],
                "category":    rec["category"],
                "skill":       s.get("skill"),
                "evidence":    s.get("evidence"),
                "depth_signal":s.get("depth_signal"),
            })

evidence_df = pd.DataFrame(evidence_flat)
print(f"Total evidence rows : {len(evidence_df):,}")
print(f"Resumes covered     : {evidence_df['resume_id'].nunique():,}")
print(f"Depth signal dist   :\n{evidence_df['depth_signal'].value_counts().to_string()}")
print()
evidence_df.sample(min(5, len(evidence_df)))

Total evidence rows : 23,494
Resumes covered     : 2,479
Depth signal dist   :
depth_signal
regular    12004
deep        8021
surface     1961
expert      1508



,resume_id,category,skill,evidence,depth_signal
1644,14743911,DESIGNER,Instructional Design,Develop training classes for customers; Develo...,deep
13571,16066857,CHEF,Mentoring,Mentored and coached employees resulting in a ...,deep
6734,10480456,HEALTHCARE,Regulatory affairs liaison,"Liaising with FDA, CHPA, CTFA, ADA, and other ...",regular
14461,24670867,FINANCE,SOX compliance,Audited merchandise invoices against internal ...,deep
19024,34544955,CONSTRUCTION,Data review,Review all technicians' reports to ensure accu...,regular


## Section 6 — Step 2: Extract Skill Requirements from JDs

Same pattern as Step 1. Saves to `jd_requirements.jsonl`, resumable.

In [29]:
CONCURRENCY_JD = 8  # higher concurrency for JDs — shorter texts, faster responses

# ── Resumability ──────────────────────────────────────────────────────────────
processed_jd_indices: set = set()
if JD_REQUIREMENTS_FILE.exists():
    with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                processed_jd_indices.add(obj["jd_index"])
            except Exception:
                pass
    print(f"Resuming: {len(processed_jd_indices)} JDs already processed.")
else:
    print("Starting fresh — no existing JD requirements file.")

Resuming: 46 JDs already processed.


In [30]:

# ── Async task: one JD → one JSONL record ─────────────────────────────────────
async def _extract_one_jd(sem: asyncio.Semaphore, jd_index: int, job_title: str, jd_text: str) -> dict:
    async with sem:
        try:
            raw_reply = await _llm_content_only(
                system=REQUIREMENT_SYSTEM,
                user=REQUIREMENT_PROMPT.format(jd_text=jd_text),
            )
        except Exception as e:
            print(f"\n[!] JD {jd_index} ({job_title}) failed: {e}")
            raw_reply = "[]"
    skills = extract_json_array(raw_reply)
    valid_skills = [
        s for s in skills
        if isinstance(s, dict)
        and s.get("requirement")
        and s["requirement"].strip().lower() != s.get("skill", "").strip().lower()
    ]
    return {"jd_index": jd_index, "job_title": job_title, "skills": valid_skills}


# ── Async orchestrator — writes & flushes each record immediately ──────────────
async def run_jd_extraction(pending_df: pd.DataFrame, out_file) -> int:
    sem = asyncio.Semaphore(CONCURRENCY_JD)
    tasks = [
        asyncio.ensure_future(
            _extract_one_jd(sem, int(idx), str(r[JOBS_TITLE_COL]), str(r[JOBS_TEXT_COL]))
        )
        for idx, r in pending_df.iterrows()
    ]
    count = 0
    pbar = tqdm(total=len(tasks), desc="JDs (concurrent)")
    for coro in asyncio.as_completed(tasks):
        record = await coro
        out_file.write(json.dumps(record) + "\n")
        out_file.flush()   # ← persists to disk immediately; safe to interrupt
        count += 1
        pbar.update(1)
    pbar.close()
    return count

print("Functions defined: _extract_one_jd, run_jd_extraction (incremental write)")


Functions defined: _extract_one_jd, run_jd_extraction (incremental write)


In [31]:

# ── SINGLE-JD TEST — identifies why results are empty ───────────────────────
# We test a row that has a meaty description (Django Developer usually does)
test_jd_row = jobs_df.iloc[1] 

async def _test_one_jd():
    sem = asyncio.Semaphore(1)
    # 1. Print description to confirm it's not empty
    print(f"--- JD Source ({test_jd_row[JOBS_TITLE_COL]}) ---")
    print(f"{test_jd_row[JOBS_TEXT_COL][:500]}...\n")
    
    # 2. Call the LLM with fixed attribute access
    try:
        raw_reply = await _llm_content_only(
            system=REQUIREMENT_SYSTEM,
            user=REQUIREMENT_PROMPT.format(jd_text=test_jd_row[JOBS_TEXT_COL]),
        )
        print(f"--- Raw LLM Reply ---")
        print(f"{raw_reply}\n")
    except Exception as e:
        print(f"!!! LLM Call Failed: {e}")
        return None

    # 3. Test the JSON extraction
    skills = extract_json_array(raw_reply)
    
    # 4. Filter validation logic
    valid_skills = [
        s for s in skills
        if isinstance(s, dict)
        and s.get("requirement")
        and s["requirement"].strip().lower() != s.get("skill", "").strip().lower()
    ]
    
    return {
        "jd_index":   int(test_jd_row.name),
        "job_title":  str(test_jd_row[JOBS_TITLE_COL]),
        "skills":     valid_skills,
        "raw_count":  len(skills)
    }

jd_result = asyncio.run(_test_one_jd())

if jd_result:
    print(f"Summary:")
    print(f"  Raw skills parsed: {jd_result['raw_count']}")
    print(f"  Valid skills:      {len(jd_result['skills'])}")
    for s in jd_result["skills"]:
        print(f"  - [{s.get('skill')}] : {s.get('requirement')[:80]}...")
    
    if len(jd_result['skills']) > 0:
        print("\n✓ SUCCESS: Pipeline is working. You can now re-run the FULL loop.")
    else:
        print("\n✗ FAIL: Parser returned 0 skills. Re-check prompt or extraction logic.")


--- JD Source (Django Developer) ---
PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)
Strong Python experience in API development (REST/RPC).
Experience working with API Frameworks (Django/flask).
Experience evaluating and improving the efficiency of programs in a Linux environment.
Ability to effectively handle multiple tasks with a high level of accuracy and attention to detail.
Good verbal and written communication skills.
Working knowledge of SQL.
JSON experience preferred.
Good knowledge in automated unit testing using PyUn...

--- Raw LLM Reply ---
[{"skill":"Python","requirement":"Strong Python experience in API development (REST/RPC).","level":"senior"},{"skill":"API Frameworks","requirement":"Experience working with API Frameworks (Django/flask).","level":"mid"},{"skill":"Linux efficiency","requirement":"Experience evaluating and improving the efficiency of programs in a Linux environment.","level":"mid"},{"skill":"SQL","requirement":"Working knowledge of SQL.","level":"jun

In [33]:

# ── FULL RUN — only execute after test above passes ───────────────────────────
pending_jd_df = jobs_df[~jobs_df.index.isin(processed_jd_indices)]
print(f"Pending: {len(pending_jd_df):,} JDs  |  Already done: {len(processed_jd_indices):,}")

if len(pending_jd_df) > 0:
    with open(JD_REQUIREMENTS_FILE, "a", encoding="utf-8") as out_f:
        count = asyncio.run(run_jd_extraction(pending_jd_df, out_f))
    print(f"\nDone. {count:,} JDs written.")
else:
    print("All JDs already processed — nothing to do.")

print(f"Output: {JD_REQUIREMENTS_FILE}")


Pending: 2,231 JDs  |  Already done: 46


JDs (concurrent):   0%|          | 0/2231 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [34]:

# ── Inspect JD requirements output ───────────────────────────────────────────
jd_flat = []
empty_jds = 0
with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        skills = rec.get("skills", [])
        if not skills:
            empty_jds += 1
            # Add a dummy record only to track the jd_index covered
            jd_flat.append({
                "jd_index":   rec["jd_index"],
                "job_title":  rec["job_title"],
                "skill":      None,
                "requirement":None,
                "level":      None,
            })
        else:
            for s in skills:
                jd_flat.append({
                    "jd_index":   rec["jd_index"],
                    "job_title":  rec["job_title"],
                    "skill":      s.get("skill"),
                    "requirement":s.get("requirement"),
                    "level":      s.get("level"),
                })

jd_df = pd.DataFrame(jd_flat)
print(f"Total rows (incl. empty) : {len(jd_df):,}")
print(f"JDs processed            : {jd_df['jd_index'].nunique():,}")
print(f"JDs with 0 skills found   : {empty_jds:,}")

if not jd_df.empty and 'skill' in jd_df.columns:
    print(f"\nLevel distribution (skills found):\n{jd_df['level'].value_counts().to_string()}")
    print()
    display(jd_df.dropna(subset=['skill']).sample(min(5, len(jd_df.dropna(subset=['skill'])))))
else:
    print("\nNo skills extracted yet. If the run is still going, wait for more data.")


Total rows (incl. empty) : 19,571
JDs processed            : 2,276
JDs with 0 skills found   : 47

Level distribution (skills found):
level
mid       11277
senior     5590
junior     2654
basic         3



,jd_index,job_title,skill,requirement,level
11178,1305,Database Administrator,Computer Literacy,computer literacy office application particula...,mid
1885,220,PHP Developer,User authentication and authorization,User authentication and authorization between ...,mid
6858,798,PHP Developer,Code versioning tools,Proficient understanding of code versioning to...,mid
13099,1521,Backend Developer,GIT version control system,"Experience with GIT version control system, au...",mid
7549,871,Full Stack Developer,PHP,back-end php front-end network communication c...,senior


## Section 7 — Step 3: Normalize Skills via O*NET (Qdrant + Nvidia LLM)

Reuses `app.services.skill_normalizer.normalize_skill` — the **exact same pipeline** the production app uses:

1. Embed raw skill name → query Qdrant `onet_skills` top-3
2. Nvidia LLM judge picks the best match (or says NONE)
3. If NONE → LLM coins a new canonical name → auto-upserted to **both Qdrant and Neo4j**

A shared `skill_cache.json` means each unique raw skill string is only normalized once across both datasets.

In [35]:
# ── Collect all unique raw skill names from both datasets ─────────────────────
all_raw_skills: set = set()

# From resume evidence
with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            raw = s.get("skill", "").strip()
            if raw:
                all_raw_skills.add(raw)

# From JD requirements
with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            raw = s.get("skill", "").strip()
            if raw:
                all_raw_skills.add(raw)

# Load existing cache to skip already-normalized skills
skill_cache: dict = {}
if SKILL_CACHE_FILE.exists():
    with open(SKILL_CACHE_FILE, "r", encoding="utf-8") as f:
        skill_cache = json.load(f)
    print(f"Cache loaded: {len(skill_cache)} skills already normalized.")

to_normalize = [s for s in sorted(all_raw_skills) if s not in skill_cache]
print(f"Total unique raw skills: {len(all_raw_skills):,}")
print(f"Still to normalize     : {len(to_normalize):,}")

Total unique raw skills: 19,116
Still to normalize     : 19,116


In [ ]:
# ── Async task: normalize one raw skill name ─────────────────────────────────
async def _normalize_one(sem: asyncio.Semaphore, raw_skill: str) -> tuple:
    """Returns (raw_skill, result_dict). Errors are caught and returned as 'error' source."""
    async with sem:
        try:
            result = await normalize_skill(raw_skill)
        except Exception as e:
            print(f"\n[!] Normalization failed for '{raw_skill}': {e}")
            result = {"matched_name": raw_skill, "canonical_id": None, "onet_level": None, "source": "error"}
    return raw_skill, result


# ── Async orchestrator — process with semaphore concurrency ──────────────
async def run_normalization(skills_list: list, sem: asyncio.Semaphore) -> dict:
    tasks = [asyncio.ensure_future(_normalize_one(sem, raw)) for raw in skills_list]
    
    results_map = {}
    pbar = tqdm(total=len(tasks), desc="Normalizing (concurrent)")
    
    # Process as they complete
    count = 0
    for coro in asyncio.as_completed(tasks):
        raw_skill, result = await coro
        results_map[raw_skill] = {
            "matched_name": result.get("matched_name", raw_skill),
            "canonical_id": result.get("canonical_id"),
            "onet_level":   result.get("onet_level"),
            "source":       result.get("source", "no_match"),
        }
        
        count += 1
        pbar.update(1)
        
        # Save cache every 50 records to prevent loss
        if count % 50 == 0:
            # Shift existing cache with new results
            temp_cache = skill_cache.copy()
            temp_cache.update(results_map)
            with open(SKILL_CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(temp_cache, f, indent=2)
                
    pbar.close()
    return results_map

# ── RUN CONCURRENTLY ────────────────────────────────────────────────────────
sem_norm = asyncio.Semaphore(CONCURRENCY_NORM)

print(f"Skills to normalize: {len(to_normalize):,}")

if to_normalize:
    new_results = asyncio.run(run_normalization(to_normalize, sem_norm))
    skill_cache.update(new_results)
    
    # Final save
    with open(SKILL_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(skill_cache, f, indent=2)

print(f"\nNormalization complete. Cache size: {len(skill_cache):,}")
sources = {}
for v in skill_cache.values():
    src = v.get("source", "unknown")
    sources[src] = sources.get(src, 0) + 1
print(f"Source breakdown: {sources}")
print(f"Saved to: {SKILL_CACHE_FILE}")


Skills to normalize: 19,116


Normalizing (concurrent):   0%|          | 0/19116 [00:00<?, ?it/s]

2026-03-21 12:43:47.963 | INFO     | app.services.skill_normalizer:normalize_skill:189 - [Normalizer] Sending 3 candidates for '*nix system and networking' to LLM judge:
1. Nagios (canonical_id: TECH_nagios, score: 0.878)
2. Dig (canonical_id: TECH_dig, score: 0.877)
3. ngrep (canonical_id: TECH_ngrep, score: 0.871)
2026-03-21 12:43:48.234 | INFO     | app.services.skill_normalizer:normalize_skill:189 - [Normalizer] Sending 3 candidates for '.NET' to LLM judge:
1. Microsoft Visual C# .NET (canonical_id: TECH_microsoft_visual_c_net, score: 0.874)
2. Microsoft .NET Framework (canonical_id: TECH_microsoft_net_framework, score: 0.869)
3. Microsoft ASP.NET (canonical_id: TECH_microsoft_asp_net, score: 0.866)
2026-03-21 12:43:48.538 | INFO     | app.services.skill_normalizer:normalize_skill:189 - [Normalizer] Sending 3 candidates for '.NET 4.5' to LLM judge:
1. Microsoft .NET Framework (canonical_id: TECH_microsoft_net_framework, score: 0.857)
2. Microsoft Visual C# .NET (canonical_id: TECH_

In [ ]:
# ── Apply normalization cache → produce normalized JSONLs ─────────────────────

def apply_norm_to_file(in_path: Path, out_path: Path, skill_key: str, text_key: str):
    """
    Read raw JSONL, apply skill_cache to each skill entry,
    and write enriched JSONL.
    
    skill_key  : key holding the raw skill name  ('skill')
    text_key   : key holding the evidence/requirement text  ('evidence' or 'requirement')
    """
    total_records = 0
    total_skills  = 0
    with open(in_path, "r", encoding="utf-8") as fin, \
         open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)
            enriched_skills = []
            for s in rec.get("skills", []):
                raw = s.get(skill_key, "").strip()
                norm = skill_cache.get(raw, {})
                enriched_skills.append({
                    **s,
                    "raw_skill":    raw,
                    "matched_name": norm.get("matched_name", raw),
                    "canonical_id": norm.get("canonical_id"),
                    "onet_level":   norm.get("onet_level"),
                    "norm_source":  norm.get("source", "no_match"),
                    # Only keep entries that have actual text
                    "_keep":        bool(s.get(text_key)),
                })
            rec["skills"] = [s for s in enriched_skills if s.pop("_keep", False)]
            fout.write(json.dumps(rec) + "\n")
            total_records += 1
            total_skills  += len(rec["skills"])
    return total_records, total_skills


res_records, res_skills = apply_norm_to_file(
    RESUME_EVIDENCE_FILE, RESUME_NORM_FILE,
    skill_key="skill", text_key="evidence",
)
jd_records, jd_skills = apply_norm_to_file(
    JD_REQUIREMENTS_FILE, JD_NORM_FILE,
    skill_key="skill", text_key="requirement",
)

print(f"Resume normalized: {res_records:,} records, {res_skills:,} valid evidence rows")
print(f"JD normalized    : {jd_records:,} records, {jd_skills:,} valid requirement rows")
print(f"\nOutputs:")
print(f"  {RESUME_NORM_FILE}")
print(f"  {JD_NORM_FILE}")

## Section 8 — Summary Statistics & Checkpoint

Validate output quality before proceeding to NB3 (pair construction + training).

In [ ]:
# ── Build flat DataFrames for final analysis ──────────────────────────────────
res_rows, jd_rows = [], []

with open(RESUME_NORM_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            res_rows.append({
                "resume_id":    rec["resume_id"],
                "category":     rec["category"],
                "raw_skill":    s.get("raw_skill"),
                "matched_name": s.get("matched_name"),
                "canonical_id": s.get("canonical_id"),
                "evidence":     s.get("evidence"),
                "depth_signal": s.get("depth_signal"),
                "norm_source":  s.get("norm_source"),
            })

with open(JD_NORM_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            jd_rows.append({
                "jd_index":     rec["jd_index"],
                "job_title":    rec["job_title"],
                "raw_skill":    s.get("raw_skill"),
                "matched_name": s.get("matched_name"),
                "canonical_id": s.get("canonical_id"),
                "requirement":  s.get("requirement"),
                "level":        s.get("level"),
                "norm_source":  s.get("norm_source"),
            })

res_norm_df = pd.DataFrame(res_rows)
jd_norm_df  = pd.DataFrame(jd_rows)

print("=" * 60)
print("STEP 3 OUTPUT SUMMARY")
print("=" * 60)
print(f"\nResume evidence rows     : {len(res_norm_df):,}")
print(f"  → onet_match           : {(res_norm_df['norm_source']=='onet_match').sum():,}")
print(f"  → llm_new (self-grown) : {(res_norm_df['norm_source']=='llm_new').sum():,}")
print(f"  → no_match / error     : {res_norm_df['norm_source'].isin(['no_match','error']).sum():,}")

print(f"\nJD requirement rows      : {len(jd_norm_df):,}")
print(f"  → onet_match           : {(jd_norm_df['norm_source']=='onet_match').sum():,}")
print(f"  → llm_new (self-grown) : {(jd_norm_df['norm_source']=='llm_new').sum():,}")
print(f"  → no_match / error     : {jd_norm_df['norm_source'].isin(['no_match','error']).sum():,}")

print(f"\nUnique canonical IDs in resumes: {res_norm_df['canonical_id'].nunique():,}")
print(f"Unique canonical IDs in JDs    : {jd_norm_df['canonical_id'].nunique():,}")

overlap = set(res_norm_df['canonical_id'].dropna()) & set(jd_norm_df['canonical_id'].dropna())
print(f"Overlapping canonical IDs      : {len(overlap):,}  ← used for pairing in NB3")

print(f"\nDepth signal distribution (resumes):")
print(res_norm_df['depth_signal'].value_counts().to_string())
print(f"\nLevel distribution (JDs):")
print(jd_norm_df['level'].value_counts().to_string())

In [ ]:
# ── Top overlapping skills (pipeline health check) ────────────────────────────
overlap_skills = (
    res_norm_df[res_norm_df['canonical_id'].isin(overlap)]
    .groupby(['canonical_id', 'matched_name'])
    .size()
    .reset_index(name='resume_count')
    .sort_values('resume_count', ascending=False)
    .head(20)
)
print("Top 20 overlapping canonical skills (appear in both resumes and JDs):")
print(overlap_skills.to_string(index=False))

print()
print("=" * 60)
print("CHECKPOINT — Ready for NB3?")
print("=" * 60)
min_overlap = 50  # conservative threshold
if len(overlap) >= min_overlap:
    print(f"✓ {len(overlap)} overlapping canonical IDs found — sufficient for pairing.")
    print("→ Proceed to Notebook 3: Pair Construction + ConFit Fine-Tuning.")
else:
    print(f"✗ Only {len(overlap)} overlapping IDs found — may need more data or looser normalization.")
    
print(f"\nFiles produced in {OUT_DIR}:")
for p in sorted(OUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45} {size_kb:>8.1f} KB")